# Lesson 54 · A `Value` computation graph — lighting the autograd fire yourself, building the heart of a computation graph out of one class

**Goal**: build the smallest beating heart of automatic differentiation, `Value`: every node remembers its value, its gradient, and how it was computed — the foundation that backpropagation stands on.

> **A foundation lesson**: first give `data / grad / _backward` an ID card each; start with only `+` and `*`, then grow the set of operators bit by bit.

🔗 **Aurora connection**: `Value` is the starting point of the Aurora ML engine (`src/aurora/ml/`, to be built in Month 2). Every training loop in Month 2 (linear layers, loss functions, optimizers) builds its computation graph out of this node and then calls `.backward()` to get the derivatives automatically.

← **Previous lesson** · [L53 · MFCCs illustrated](../5_audio_dsp/L53_visual_mfcc.ipynb)

> Last lesson covered **MFCCs illustrated**: waveform → spectrogram → mel spectrum → cepstral coefficients, layer by layer.
> This lesson moves into deep learning and builds the rules of automatic differentiation from the ground up.

## Review bridge · the L24 chain rule (5 minutes, don't skip it)

- Formula: `z = f(g(x))` → `dz/dx = (dz/dg) · (dg/dx)`
- Worked example: `z = x*y + y²`, `x=3, y=2` → `dz/dx = 2`, `dz/dy = 7`
- What this lesson does: hand those two multiplications over to `Value.backward()` so they happen automatically

In [ ]:
x, y = 3.0, 2.0
z = x * y + y**2
dz_dx = y
dz_dy = x + 2 * y
assert z == 10 and dz_dx == 2 and dz_dy == 7
print("Hand-computed chain rule agrees with the L54 assertions")

## 📋 Cheat sheet: partial derivatives of multivariable functions

At school you only learned derivatives of single-variable functions: `y = f(x) → dy/dx`.
Now we have multivariable functions, such as `z = x*y + y²`, with two inputs x and y. Which one do we differentiate with respect to?

**Key idea**: when you take the partial derivative with respect to one variable, treat every other variable as a constant. The symbol `∂z/∂x` means "the partial derivative of z with respect to x".

**Common rules** (exactly like single-variable derivatives):

| Function | Partial w.r.t. x | Partial w.r.t. y |
|---|---|---|
| `c` (a constant) | 0 | 0 |
| `x` | 1 | 0 |
| `y` | 0 | 1 |
| `x + y` | 1 | 1 |
| `x * y` | **y** | **x** |
| `x²` | **2x** | 0 |
| `y²` | 0 | **2y** |
| `x*y + y²` | **y** | **x + 2y** |

**Deriving the partials of `z = x*y + y²`** (worked through):
- `∂z/∂x = ∂(xy)/∂x + ∂(y²)/∂x = y + 0 = y`  ← in the first term y is a constant when we differentiate by x; the second term has no x in it
- `∂z/∂y = ∂(xy)/∂y + ∂(y²)/∂y = x + 2y`  ← in the first term x is a constant when we differentiate by y; the second term gives 2y

**Plug in numbers to check** (`x=3, y=2`):
- `∂z/∂x = y = 2` ✓
- `∂z/∂y = x + 2y = 3 + 4 = 7` ✓

## Side-by-side comparison

| | L53 MFCC | L54 Value |
|---|---|---|
| Data type | `ndarray (T, 13)` | scalar `float` |
| Direction | forward only | forward + backward |
| Goal | a compressed representation | compute `dL/d(parameter)` automatically |
| Library | `aurora.audio` | written by hand in this lesson (swapped for PyTorch in L59) |

Memory hook: L53 is "feature extraction"; L54 is "letting parameters learn from their mistakes".

## This lesson's story: what is PyTorch's core magic trick?

You write `loss.backward()`, and PyTorch works out the gradient of every parameter for you. What's going on behind it?

**The chain rule from calculus**: if `z = f(g(x))`, then `dz/dx = dz/dg · dg/dx`. A neural network is just one long chain of composed functions; apply the chain rule layer after layer and you can work "backwards" from the gradient at the output to the gradient of every parameter — this is called **backpropagation**.

**What is tanh (the hyperbolic tangent)?** Lots of students get stuck here because they never met it at school. tanh is a "squashing function": it squeezes any number into the range -1 to 1:
- Definition: `tanh(x) = (e^x - e^(-x)) / (e^x + e^(-x))` (using Euler's number e)
- Derivative (no need to memorize it, the course will give it to you): `d(tanh(x))/dx = 1 - tanh(x)²`
- Intuition: like the sigmoid function, it "squeezes" very large and very small values into a finite range — neural networks love activation functions like this

This lesson uses tanh only to show "a reasonable nonlinear function". Concretely:

**The problem**: the loss function `L = tanh(a*b + b²)` has 2 parameters, and writing its partial derivatives by hand means applying the chain rule layer by layer:
```
∂L/∂a = (1 - tanh²(a*b+b²)) · ∂(a*b+b²)/∂a
      = (1 - tanh²(a*b+b²)) · b
∂L/∂b = (1 - tanh²(a*b+b²)) · ∂(a*b+b²)/∂b
      = (1 - tanh²(a*b+b²)) · (a + 2b)
```
That's for 2 parameters. A neural network has millions of parameters — there's no way to write each one out by hand.

**Karpathy's micrograd insight**: the Python object graph *is* the computation graph. Every time you write `c = a + b`, you quietly build a `+` node:

```
a ──┐
    ├──[+]──► c      c.data = a.data + b.data
b ──┘         c._backward() → a.grad += 1*c.grad, b.grad += 1*c.grad
```

Backpropagation = start at the output node and walk the directed acyclic graph (DAG) in reverse, passing the gradient back one layer at a time, from the end to the start.

Task for this section: implement a minimal `Value` class that supports `__add__`, `__mul__`, `__pow__` and `backward()`.

## 🤔 Why did engineers invent this?

- **What happens without it?** Every time you add a layer to the network, you have to derive a fresh set of partial-derivative formulas by hand; switch the activation function and you start all over again — research simply couldn't iterate.
- **What real problem does it solve?** A `Value` object ties together "the number + how it was computed", so the computation graph **builds itself while you write the forward code**, and the gradients can then flow back at the press of a button — this is exactly the minimal kernel of automatic differentiation in PyTorch / TensorFlow.
- **Where does it come up again?** L55 adds operators, L56 implements `backward()`, L57 builds an MLP, L60 switches to PyTorch autograd — all of them stand on the shoulders of this `Value` class.

## ⚠️ Common misconceptions

> Don't read a `Value`'s `grad` as "the value of this number itself". It stores **the partial derivative of the loss with respect to this node**, `dL/d(node)`: right after the forward pass it is always 0, and it only means something once `backward()` has finished. And don't overwrite `grad` with `=` — the same node can be reused in several places (e.g. `b` appears twice in `a*b + b**2`), so gradients must accumulate with `+=`.

In [ ]:
# Pure Python in this lesson, no numpy needed
# (this lesson uses no numpy features at all — the Value class only uses Python built-in types)

## 1. The three core fields of `Value`

Every `Value` node carries three things:

- `data`: the scalar value from the forward computation, e.g. `3.14`
- `grad`: the partial derivative of the final loss L with respect to this node, `dL/d(self)`, **initialized to 0.0** (it accumulates; it is never overwritten)
- `_backward`: a **closure** that, when called, passes the current node's `grad` on to its input nodes according to the chain rule

`grad` starts at `0.0` rather than `None` because the same node can be used along several paths (e.g. `b` appears twice in `a*b + b**2`), and its gradient has to be **added up**, not overwritten.

In [ ]:
# Demo: why grad must accumulate, with concrete numbers
# Scenario: b is used twice (in a*b and in b²), so the gradients from both paths must be added

# Concrete values: a=3, b=2
a_val = 3.0
b_val = 2.0

# Forward pass
L_val = a_val * b_val + b_val ** 2  # L = 3*2 + 4 = 10

# Backpropagation by hand:
# by the rules for multivariable partial derivatives
# ∂L/∂b = ∂(a*b)/∂b + ∂(b²)/∂b = a + 2b = 3 + 2*2 = 7

# But which two paths does that 7 come from?
# Path 1: L depends on b through the (a*b) term → gradient contribution = a = 3
# Path 2: L depends on b through the (b²) term → gradient contribution = 2*b = 2*2 = 4
# Total gradient = 3 + 4 = 7 ✓

b_grad_from_path1 = a_val        # partial of a*b with respect to b
b_grad_from_path2 = 2 * b_val    # partial of b² with respect to b

total_b_grad = b_grad_from_path1 + b_grad_from_path2

print(f"Numeric demo (a={a_val}, b={b_val}):")
print(f"  L = a*b + b² = {L_val}")
print(f"  Path 1 (a*b): ∂(a*b)/∂b = a = {b_grad_from_path1}")
print(f"  Path 2 (b²): ∂(b²)/∂b = 2*b = {b_grad_from_path2}")
print(f"  Accumulate: b.grad = {b_grad_from_path1} + {b_grad_from_path2} = {total_b_grad}")
print(f"  ✓ matches ∂L/∂b = a + 2*b = {a_val + 2*b_val}")
print()

# Key point: what happens if we overwrite with = instead of accumulating with +=
print("If we wrongly use '=' instead of '+=':")
print(f"  First time: b.grad = {b_grad_from_path1}  (from a*b)")
print(f"  Second time: b.grad = {b_grad_from_path2}  (from b², overwriting the first!)")
print(f"  Wrong result! It should be {total_b_grad}, but we got {b_grad_from_path2}")

## 2. The computation graph is a DAG

Every time you write `c = a + b`, you quietly create a **directed acyclic graph (DAG)**:

```
a ──┐
    ├─[+]─► c
b ──┘
```

- **Nodes**: each `Value` object
- **Edges**: operators (`+`, `*`, `**`, …); edges point the way data flows (forward)
- **`_prev`**: each node stores the set of its direct input nodes (`{a, b}`); during the backward pass we trace back through the graph using it

Having no cycles is crucial — if there were a cycle, the gradient would loop forever. In a feed-forward network, data flows one way from the input to the loss, so it is naturally a DAG.

In [ ]:
# Simulate the DAG structure with a dict to show the topological relationships
# (the real Value class is further down; this only shows the idea of the graph structure)
graph = {
    "c": {"op": "+", "inputs": ["a", "b"]},
    "d": {"op": "*", "inputs": ["c", "b"]},
}
def topo(node, graph, visited=None, order=None):
    if visited is None: visited, order = set(), []
    if node not in visited:
        visited.add(node)
        for inp in graph.get(node, {}).get("inputs", []):
            topo(inp, graph, visited, order)
        order.append(node)
    return order

print("Topological order (forward):", topo("d", graph))
print("Backpropagation order:", list(reversed(topo("d", graph))))

## The Python mutable-default-argument "trap" and the right way to do it

Many people are puzzled when they see the signature of `topo()`: why not just write `visited=set(), order=[]`?

**The dangerous, wrong version**:
```python
def topo(node, graph, visited=set(), order=[]):  # ❌ dangerous!
    if node not in visited:
        visited.add(node)
        order.append(node)
    return order
```

The problem: Python creates **one single object** for the default value **when the function is defined** (not when it is called). Every call shares that object!

```python
result1 = topo("a", {})     # visited and order are created once
result2 = topo("b", {})     # reuses the same visited and order! the results get mixed together
print(result1)              # ["a", "b"] (wrong! it should only be "a")
```

**The right way**:
```python
def topo(node, graph, visited=None, order=None):
    if visited is None:
        visited = set()
    if order is None:
        order = []
    # ... rest of the code
```

This way every call creates a fresh `set()` and `[]`, and calls don't interfere with each other. This is the standard Python idiom.

In our Value backpropagation this matters a lot: every call to `backward()` needs a brand-new `visited` set to track which nodes have already been handled.

## Topological sort and the order of backpropagation

Why does backpropagation have to call `_backward()` in a particular order (and not just any order)?

**Analogy**: picture a factory production line:
- Forward: raw materials → Workshop A → Workshop B → Workshop C → finished product
- Inspecting backwards: you must check the finished product first, then go back to Workshop C (its output is the finished product), then Workshop B, and finally Workshop A

If you inspect Workshop A first, before Workshop B's output has been checked, how would you know whether the problem was A's fault or B's input that was set up wrong?

**Mathematically**: we need to guarantee that when a node's `_backward()` is called, the gradients of its output nodes (its "downstream" in the graph) have all been computed already. Only then does `out.grad` hold a valid value.

**Definition of a topological sort**: run a DFS (depth-first search) over the graph, "handling the input nodes first, then adding yourself"; the sequence you get has exactly this property. Reverse that sequence and you have the correct order for backpropagation.

Example (graph):
```
    a ──┐
        ├──[+]──► c ──┐
    b ──┘            │
                     ├──[*]──► e
    d ────────────────┘
```

Forward topological order (DFS post-order): a, b, c, d, e
Backpropagation order (reversed): e, d, c, b, a

So when c's `_backward()` is handled, e's gradient is already there (e depends on c's output).

## How the chain rule shows up in code

In maths, the chain rule is: `dL/da = (dL/dout) · (dout/da)`.

What does that turn into in backpropagation code?

Say `out = a + b` (addition). During the backward pass:
- `out.grad` already has a value (say 1.0, or something more complicated) — that is `dL/dout`
- The local partial derivatives of addition: `dout/da = 1`, `dout/db = 1`
- By the chain rule:
  - `da_contrib = (dL/dout) * (dout/da) = out.grad * 1 = out.grad`
  - `db_contrib = (dL/dout) * (dout/db) = out.grad * 1 = out.grad`
- So `a.grad += out.grad` and `b.grad += out.grad`

Now multiplication, `out = a * b`:
- Local partial derivatives: `dout/da = b`, `dout/db = a`
- Chain rule:
  - `da_contrib = (dL/dout) * (dout/da) = out.grad * b = out.grad * b.data`
  - `db_contrib = (dL/dout) * (dout/db) = out.grad * a = out.grad * a.data`
- So `a.grad += b.data * out.grad` and `b.grad += a.data * out.grad`

**Key point**: that multiplication by `out.grad` inside `_backward` *is* the one multiplication of the chain rule!

## 3. Why use a Python class

The core insight of Karpathy's micrograd: **the Python object graph = the computation graph**.

You don't need a special graph database. Just overload operators like `__add__` and `__mul__` — every operation automatically creates a new node and records the parent–child relationship. Python's garbage collector takes care of memory, the `_prev` set maintains the graph structure, and the `_backward` closure captures every local variable it needs.

Compare that with differentiating by hand: suppose `L = x * y + y**2`; by hand you write `dL/dx = y`, `dL/dy = x + 2y`. When a network has a million parameters, doing that by hand is impossible — with `Value`, each operator only needs to know its own **first-order local partial derivative**, and backpropagation combines them automatically.

## A crash course in Python closures

You may have heard of "closures" before, but how exactly are they used in `_backward`?

**The simplest way to understand it**: a function can be defined inside another function and "remember" the outer function's variables, even after the outer function has finished running. Example:

```python
def make_adder(n):
    def add_x(x):        # inner function
        return x + n     # uses the outer function's parameter n
    return add_x

add_5 = make_adder(5)
print(add_5(3))          # prints 8 (3+5)
```

Here `add_x` is a closure — it has "closed over" the variable `n`.

**How it's used in the Value class**:

```python
def __add__(self, other):
    out = Value(...)

    # this inner function is a closure: it "remembers" self, other and out
    def _backward():
        self.grad += out.grad
        other.grad += out.grad

    out._backward = _backward
    return out
```

When `__add__` finishes and returns `out`, the `_backward` closure still "remembers" which objects `self`, `other` and `out` are, even if they get used elsewhere later. When backpropagation calls `out._backward()`, the gradients of those three objects get updated correctly.

**Why not use a lambda?** Try it:

```python
# ❌ wrong attempt
_backward = lambda: (self.grad += out.grad)
# SyntaxError: `+=` is a statement and can't appear in the expression a lambda returns
```

So you need a real function body, built from statements rather than expressions.

In [ ]:
# Demo of the syntactic sugar of operator overloading (a pseudo-code comparison)
print("Differentiating by hand (doesn't scale):")
print("  L = x*y + y**2")
print("  dL/dx = y")
print("  dL/dy = x + 2*y")
print()
print("The Value approach: each operator writes its local partial derivative once; the chain rule combines them automatically")
print("  _backward of __mul__: a.grad += b.data * out.grad")
print("                         b.grad += a.data * out.grad")

## Half-step exercise A · forward pass only (15 minutes)

First complete only `__init__`, `__add__` and `__mul__`; don't rush into `__pow__` and `backward()` yet.
The aim is to get the forward values working first and add backpropagation afterwards, so you don't run into two hard problems at once.

## 4. ✏️ Implement `class Value`

**The three core fields**:

| Field | Meaning | Initial value |
|---|---|---|
| `data` | scalar forward value | passed to the constructor |
| `grad` | gradient of the final loss | 0.0 (waiting for backpropagation to fill it in) |
| `_backward` | this operator's backward function | `lambda: None` |

**A four-step implementation route**:

| Step | Method | Backward formula |
|---|---|---|
| 1 | `__init__` | — |
| 2 | `__add__` | `a.grad += out.grad`, `b.grad += out.grad` |
| 3 | `__mul__` | `a.grad += b.data * out.grad`, `b.grad += a.data * out.grad` |
| 4 | `backward()` | topological sort → call each node's `_backward()` in reverse order |

**Acceptance criteria**:
- `(Value(2) + Value(3)).data == 5`
- `L = a*b + b**2`, `L.backward()`, `a.grad == b.data` (the partial w.r.t. a = b)

**Groundwork**: do Half-step exercise A above first, then add `__pow__` and `backward()`; get the forward assertions passing first and the backward assertions after.

## The core fields of the Value class in detail

When you implement `__init__`, you need to store these fields:

| Field | Type | Meaning | Initial value or source |
|---|---|---|---|
| `data` | `float` | scalar value of the forward computation | constructor argument, converted to float |
| `grad` | `float` | partial derivative of the loss L, `dL/dself` | starts at 0.0 (filled in later by backpropagation) |
| `_backward` | `function` | closure called during backpropagation | starts as the empty function `lambda: None`; reassigned inside `__add__`/`__mul__` etc. |
| `_prev` | `set` | this node's "parents" (its direct inputs) | converted from the `_children` argument |
| `_op` | `str` | name of the operator that produced this node | e.g. `"+"`, `"*"`, `"**"`; used for debugging |

**The life cycle of the `_children` argument**:

When you create the output node inside `__add__`:
```python
def __add__(self, other):
    out = Value(
        self.data + other.data,
        _children=(self, other),      # ← says who this node's "parents" are
        _op="+"                       # ← records how it was produced
    )
    # ... then define _backward ...
    return out
```

Inside `__init__`:
```python
def __init__(self, data, _children=(), _op=""):
    self.data = float(data)
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)    # ← turn _children into a set
    self._op = _op
```

## Deriving the operators' _backward: power and multiplication

### Power function

**Local partial derivative** (you met this at school): if `out = x^n`, then `dout/dx = n * x^(n-1)`

Example: `y = x³` → `dy/dx = 3*x²`; at x=2, `dy/dx = 3*4 = 12`

**In backpropagation**: let `out = self^other` (where `other` is a constant, such as `2` or `3`); then:
```
dL/dself = (dL/dout) * (dout/dself)
         = out.grad * other * self.data^(other-1)
```

So `_backward` is written as:
```python
def _backward():
    self.grad += other * self.data**(other-1) * out.grad
```

### Multiplication

**Local partial derivatives**: if `out = a * b`, then:
- `dout/da = b`
- `dout/db = a`

**In backpropagation**:
```
dL/da = (dL/dout) * (dout/da) = out.grad * b.data
dL/db = (dL/dout) * (dout/db) = out.grad * a.data
```

So `_backward` is written as:
```python
def _backward():
    a.grad += b.data * out.grad
    b.grad += a.data * out.grad
```

**Why `b.data` and not `b`?** Because `b` is a Value object, and we need its numeric value `b.data` to multiply the gradient by.

### Vocabulary at a glance

| Term | Meaning |
|---|---|
| `out.grad` | partial derivative of the loss w.r.t. this node's output, coming from downstream |
| local partial derivative | this node's own partial derivative (looking only at this operator, not at anything downstream) |
| `a.grad +=` | accumulate the gradient (it may arrive along several paths) |

## Why does backward() set self.grad = 1.0 first?

Backpropagation needs a "starting" gradient. When we call `loss.backward()`:
- what we want is "the partial derivative of the loss with respect to every parameter"
- and the partial derivative of the loss with respect to itself is 1 (mathematically obvious: `dL/dL = 1`)

So we start from `loss.grad = 1.0`, and the chain rule multiplies that 1.0 down through the layers, until the final gradient has accumulated at every leaf node.

**A concrete example** (L = (a * b) + b², a=3, b=2):

```
Forward:
  a=3.0 ──┐
          ├──[*]──► tmp1=6.0 ──┐
  b=2.0 ──┘                      │
                                  ├──[+]──► L=10.0
          b=2.0 ──┐              │
                  ├──[²]──► tmp2=4.0 ──┘

Backward (calling _backward in reverse order):
  1) L.grad = 1.0  ← initialize the starting point
  2) _backward of [+]:
     tmp1.grad += 1.0 * 1 = 1.0
     tmp2.grad += 1.0 * 1 = 1.0
  3) _backward of [*] (which computed tmp1=a*b):
     a.grad += b.data * tmp1.grad = 2.0 * 1.0 = 2.0
     b.grad += a.data * tmp1.grad = 3.0 * 1.0 = 3.0
  4) _backward of [²] (which computed tmp2=b**2):
     b.grad += 2 * b.data * tmp2.grad = 2 * 2.0 * 1.0 = 4.0

  Final: a.grad = 2.0, b.grad = 3.0 + 4.0 = 7.0
```

See it? `L.grad = 1.0` is like the "multiplier" in the very first multiplication; then, as we travel backwards through the computation graph, each layer multiplies in its own local partial derivative.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=""):
        # ✏️ TODO: store data (as a float), grad=0.0, _backward=lambda:None
        #          _prev=set(_children), _op=_op
        raise NotImplementedError("Implement __init__: set self.data, self.grad, self._backward, self._prev, self._op")

    def __repr__(self):
        # use getattr so the stub state (before __init__ is implemented) doesn't raise AttributeError
        return f"Value(data={getattr(self, 'data', '?')}, grad={getattr(self, 'grad', '?')})"

    def __add__(self, other):
        # ✏️ TODO: wrap other if it isn't a Value; create out; define _backward (both partials of addition are 1)
        raise NotImplementedError("Implement __add__: wrap other, create out, set _backward")

    def __radd__(self, other): return self + other

    def __mul__(self, other):
        # ✏️ TODO: product rule _backward
        raise NotImplementedError("Implement __mul__: wrap other, create out, set _backward using the product rule")

    def __rmul__(self, other): return self * other

    def __pow__(self, other):
        # ✏️ TODO: partial derivative of a power: other * self.data**(other-1)
        assert isinstance(other, (int, float)), "the exponent must be a scalar"
        raise NotImplementedError("Implement __pow__: create out, set _backward (partial = other * self.data**(other-1))")

    def __neg__(self):  return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __truediv__(self, other): return self * other**-1

    def backward(self):
        # ✏️ TODO: topologically sort the whole graph and call _backward in reverse order; set self.grad=1.0 first
        # Hint: follow the topo() function in the DAG demo above: DFS + a visited set to collect nodes in post-order, then reverse
        raise NotImplementedError("Implement backward: first self.grad=1.0, then topologically sort and call each node's _backward in reverse order")

In [ ]:
# Forward half-step: as soon as __init__ / __add__ / __mul__ are done, this part should pass
a = Value(2.0)
b = Value(3.0)
c = a + b
assert c.data == 5.0, f"expected 5.0, got {c.data}"
print("✅ a + b forward correct:", c)

d = a * b
assert d.data == 6.0, f"expected 6.0, got {d.data}"
print("✅ a * b forward correct:", d)

# Full backward: if __pow__ / backward aren't finished yet, this part prints a friendly note
try:
    a2 = Value(2.0)
    b2 = Value(3.0)
    out = a2 * b2
    out.grad = 1.0
    out._backward()
    assert a2.grad == 3.0, f"expected a.grad=3.0, got {a2.grad}"
    assert b2.grad == 2.0, f"expected b.grad=2.0, got {b2.grad}"
    print("✅ __mul__ _backward correct: a.grad=", a2.grad, " b.grad=", b2.grad)

    a3 = Value(4.0)
    b3 = Value(2.0)
    L = a3 * b3 + b3 ** 2   # L = 4*2 + 4 = 12; dL/da=2, dL/db=4+4=8
    L.backward()
    assert a3.grad == 2.0, f"expected dL/da=2.0, got {a3.grad}"
    assert b3.grad == 8.0, f"expected dL/db=8.0, got {b3.grad}"
    print("✅ L=(a*b + b**2) backward() correct: a.grad=", a3.grad, " b.grad=", b3.grad)
except (NotImplementedError, TypeError):
    print("⬜ backward / __pow__ not finished yet — finish the forward half-step first and you can carry on.")

## 5. Parameter experiment: by hand vs automatic differentiation

Build `L = (a * b) + b**2` with `a=3.0, b=2.0`.

**By hand**:
- `L = 3*2 + 2**2 = 10`
- `dL/da = b = 2`
- `dL/db = a + 2*b = 3 + 4 = 7`

**What to expect**: after calling `L.backward()`, `a.grad==2.0` and `b.grad==7.0`.

Change the values of `a` and `b` and watch how the gradients change, linearly or nonlinearly, with the inputs.

## The full derivation: the gradient of L = (a*b) + b²

This is the single most important example. Let's derive the gradient formulas all the way through.

**Step 1: pin down the function**
```
L = (a*b) + b²
```

**Step 2: take the partial derivatives with respect to a and b** (using the rules for multivariable partials)

With respect to a:
```
∂L/∂a = ∂(a*b)/∂a + ∂(b²)/∂a
      = b + 0                   (first term: with b constant, it is a's coefficient; second term has no a)
      = b
```

With respect to b (here the chain rule has to combine two paths):
```
∂L/∂b = ∂(a*b)/∂b + ∂(b²)/∂b
      = a + 2*b                 (first term: with a constant, it is b's coefficient; second term: 2*b)
```

**Step 3: plug in numbers to check** (a=3, b=2)
```
L = 3*2 + 2² = 6 + 4 = 10 ✓
∂L/∂a = b = 2 ✓
∂L/∂b = a + 2*b = 3 + 4 = 7 ✓
```

**Step 4: understand gradient accumulation**

Where does b's gradient of 7 come from?
- Path 1: the contribution of the (a*b) term in L to b = a = 3
- Path 2: the contribution of the (b²) term in L to b = 2*b = 4
- Total: 3 + 4 = 7

In the backpropagation code, this is exactly why we accumulate with `+=` instead of overwriting with `=`!

In [ ]:
# Parameter experiment: change the values of a, b and check the gradient formulas
for a_val, b_val in [(3.0, 2.0), (1.0, 5.0), (-2.0, 3.0)]:
    a = Value(a_val); b = Value(b_val)
    L = a * b + b ** 2
    L.backward()
    # expected values, computed by hand
    expected_da = b_val
    expected_db = a_val + 2 * b_val
    print(f"a={a_val}, b={b_val} → L={L.data:.1f} | "
          f"a.grad={a.grad:.1f} (expected {expected_da:.1f}) "
          f"b.grad={b.grad:.1f} (expected {expected_db:.1f})")
    assert abs(a.grad - expected_da) < 1e-9
    assert abs(b.grad - expected_db) < 1e-9
print("✅ All parameter combinations check out")

## 🎯 The payoff later

The `Value` class you built yourself today comes back in **L55 / L56 / L60** — L55 fills in its operators, L56 teaches it to run `backward()` automatically, and by L60 you will recognize the very same `.grad` / `.backward()` design in PyTorch at a glance. Once you have written it from scratch, the framework is no longer a black box to you.

## Wrap-up

The `Value` class builds a computation graph through three overloaded operators, `__add__`, `__mul__` and `__pow__`. Each operator writes its own local partial derivative into its `_backward` closure, and `backward()` fires every `_backward` in reverse topological order, so that each leaf node's `.grad` ends up holding `dL/d(leaf)`. What this machinery produces is **scalar gradients**, matching the core concept of the Aurora ML engine (`src/aurora/ml/` is a planned module that doesn't exist yet; for now the automatic differentiation (autograd) implementation is demonstrated directly in the notebook). The next lesson (**L55**) completes the `__pow__`, `tanh`, `relu` and `exp` operators, filling out the computation graph's operator library; building `Neuron → Layer → MLP` comes in L57, and the real training loop in L58.

## ✏️ Closed-book derivation check cell — backpropagation through a computation graph

**Rule: close every cell above and do the derivation below from memory alone.**

**Problem**: given the expression $z = x \cdot y + y^2$ with $x = 3, y = 2$:

1. Draw the forward computation graph (label the value of every intermediate node)
2. Write out the chain-rule expansion by hand and find $\partial z / \partial x$ and $\partial z / \partial y$
3. Check your answers: $\partial z / \partial x = ?$, $\partial z / \partial y = ?$

(Write your derivation here...)

In [ ]:
# Check: use this notebook's Value if possible; if it isn't finished yet, fall back to numerical differences for comparison (pure Python, no extra dependencies)
try:
    x = Value(3.0)
    y = Value(2.0)
    z = x * y + y ** 2
    z.backward()
    dz_dx = x.grad
    dz_dy = y.grad
except (NotImplementedError, TypeError):
    # central difference: the slope between f(x+h) and f(x-h) approximates the derivative (almost exact for a quadratic)
    f = lambda x, y: x * y + y ** 2
    h = 1e-6
    dz_dx = (f(3.0 + h, 2.0) - f(3.0 - h, 2.0)) / (2 * h)
    dz_dy = (f(3.0, 2.0 + h) - f(3.0, 2.0 - h)) / (2 * h)

# z = xy + y^2 → dz/dx = y = 2, dz/dy = x + 2y = 3 + 4 = 7
assert abs(dz_dx - 2.0) < 1e-6, f"dz/dx = {dz_dx}, expected 2"
assert abs(dz_dy - 7.0) < 1e-6, f"dz/dy = {dz_dy}, expected 7"
print(f"✅ dz/dx = {dz_dx} ✓   dz/dy = {dz_dy} ✓")

In [ ]:
# ✏️ Self-review for this lesson
l54_review = {
    "chain_rule_understood":   None,  # understand the chain rule: dz/dx=dz/dg·dg/dx? True/False
    "value_class_impl":        None,  # Value.__add__/__mul__/__pow__/backward all implemented? True/False
    "backprop_dag_order":      None,  # understand backpropagation = calling _backward in reverse topological order? True/False
    "grad_accumulation":       None,  # understand that grad must accumulate when one node is used several times? True/False
    "whiteboard_passed":       None,  # whiteboard challenge (backprop through the computation graph) passed? True/False
}

unfilled = [k for k, v in l54_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l54_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L54 fully cleared! On to L55: completing the operators')

---

→ **Next lesson** · [L55 · Completing the operators](L55_forward_pass.ipynb)

> Next lesson covers **completing the operators**: implementing the pow, relu, tanh and exp nodes, and the full computation graph.